[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# Directories &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.


In [1]:

from pathlib import Path
from collections import Counter
import os
import shutil

scratch = Path("scratch")
root = scratch / "project"

for folder in ["data/2025", "data/2026", "reports", "__pycache__"]:
    (root / folder).mkdir(parents=True, exist_ok=True)

for name in ["README.md", "data/notes.txt", "data/2025/jan.csv", "data/2025/feb.csv",
             "data/2026/jan.csv", "reports/summary.md", "reports/summary.pdf",
             "__pycache__/cache.pyc"]:
    (root / name).write_text("x" * (len(name) * 10))

print("tree ready under", root)


tree ready under scratch/project


**1.** List everything directly inside `root`, sorted, saying whether each is a file or a
folder.


In [2]:

for path in sorted(root.iterdir()):
    print(f"{'dir ' if path.is_dir() else 'file'} {path.name}")


file README.md
dir  __pycache__
dir  data
dir  reports


`iterdir` goes one level deep, so the files inside `data` and `reports` do not appear.


**2.** Find every `.md` file at any depth.


In [3]:

for path in sorted(root.rglob("*.md")):
    print(path.relative_to(root))


README.md
reports/summary.md


`rglob` rather than `glob`, because `README.md` is at the top and `summary.md` is two levels
down. `glob("*.md")` would have found only the first.


**3.** Find every file whose name starts with `s`, at any depth.


In [4]:

for path in sorted(root.rglob("s*")):
    if path.is_file():
        print(path.relative_to(root))


reports/summary.md
reports/summary.pdf


The `is_file()` check matters: a folder named `something` would match the pattern too. `rglob`
returns folders as well as files, and nothing in a pattern distinguishes them.


**4.** Walk the tree with `os.walk`, skipping `__pycache__`, and print each folder with its
file count.


In [5]:

for folder, subdirs, filenames in os.walk(root):
    subdirs[:] = [d for d in subdirs if d != "__pycache__"]
    print(f"{Path(folder).relative_to(root) or '.'}  {len(filenames)} files")


.  1 files
data  1 files
data/2025  2 files
data/2026  1 files
reports  2 files


`subdirs[:] = [...]` is what prunes the walk. Writing `subdirs = [...]` would have created a new
local list and `os.walk` would have descended into `__pycache__` anyway.


**5.** Build a list of records for every `.csv` file, each with its path and size, and print
the total.


In [6]:

records = [
    {"path": str(p.relative_to(root)), "bytes": p.stat().st_size}
    for p in sorted(root.rglob("*.csv"))
]

for r in records:
    print(f"  {r['path']:<22} {r['bytes']:>5}")

print("total:", sum(r["bytes"] for r in records), "bytes across", len(records), "files")


  data/2025/feb.csv        170
  data/2025/jan.csv        170
  data/2026/jan.csv        170
total: 510 bytes across 3 files


The path is stored, not the name. That decision is what task 6 is about.


**6.** Count how many files share a name with another file somewhere else in the tree.


In [7]:

names = Counter(p.name for p in root.rglob("*") if p.is_file())
duplicates = {name: count for name, count in names.items() if count > 1}

print("duplicated names:", duplicates)
print("files involved:  ", sum(duplicates.values()))


duplicated names: {'jan.csv': 2}
files involved:   2


Two files called `jan.csv`, in different year folders. A dictionary keyed by name would hold one
of them and give no sign the other existed.

This is worth checking on any folder you did not create, before building a lookup from it.


In [8]:

shutil.rmtree(scratch)

print("cleaned up:", not scratch.exists())


cleaned up: True


---

&#8592; **Back to:** [Directories](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/07-directories.ipynb)  &nbsp;&middot;&nbsp;  [Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)
